# Session 1 — Setup and First Words

### Study Buddy · Session 1 of 8

By the end of these eight sessions you'll have built an assistant that has read **your own
notes** and answers questions about them — with sources, and with the honesty to say "that
isn't in your documents" when it isn't.

Today is the foundation: get the machine working, translate the JavaScript you already know
into Python, and make a model talk.

## TODAY:

- **Part A:** is your setup working?
- **Part B:** Python, for people who already know JavaScript
- **Part C:** your first model call
- **Part D:** system prompts — giving a model a job

<div style="border-left: 6px solid #900; background: #fff4f4; padding: 12px 16px; margin: 12px 0;">
<h3 style="color:#900; margin-top:0;">🛑 Before you run anything</h3>
<p style="color:#900; margin-bottom:0;">
Look at the <b>top right</b> of this notebook. If it says <b>Select Kernel</b>, click it →
<b>Python Environments...</b> → pick the one showing <code>.venv</code> with a star.<br/><br/>
You must do this <b>for every notebook you open</b>. Nearly every "but I installed it!"
problem in this workshop is this and nothing else.
</p>
</div>

### How a notebook works

A notebook is a list of **cells**. Click a code cell, press **Shift + Enter** to run it.

Two rules:

1. **Run every cell from the top, in order.** Cells share memory — a cell can only use a
   variable if the cell that created it has already run.
2. **When things stop making sense, restart the kernel and run everything again.** This
   fixes a genuinely surprising proportion of problems.

## PART A: Is your setup working?

In [ ]:
# imports

import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage, HumanMessage

# A red error here means either the kernel is wrong (see the box above) or
# pip install didn't finish. Raise your hand - don't continue.

In [ ]:
# Load the .env file and check the key looks sensible.
#
# override=True means "re-read the file even if something was loaded earlier" - without it,
# editing .env and re-running would silently keep using the old value.

load_dotenv(override=True)
api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    print("❌ No API key found. Is there a file called exactly '.env' next to this notebook?")
elif api_key.strip() != api_key:
    print("❌ A key was found, but it has a space or tab at the start or end. Delete it.")
else:
    print(f"✅ API key found, begins {api_key[:8]}... - looks good.")

In [ ]:
# Check the local embedding model is already downloaded.
# This is the one thing that takes real time if it hasn't been cached - if this cell
# takes several minutes, that's it downloading ~900 MB. It only happens once.

from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
print("✅ Embedding model ready (we'll find out what this does in Session 4)")

## PART B: Python, for people who already know JavaScript

You know what a variable is, what a loop does, what a function is for. **None of that is
being taught again.** What's actually different is mostly spelling.

<div style="border-left: 6px solid #06c; background: #f2f7ff; padding: 12px 16px; margin: 12px 0;">
<h3 style="color:#06c; margin-top:0;">🔄 The whole translation table</h3>
<pre style="color:#06c; margin-bottom:0; background:none; border:none;">
// JavaScript                        # Python
let name = "Ada";                    name = "Ada"
const MAX = 10;                      MAX = 10          (caps = "don't change me")
`Hello ${name}`                      f"Hello {name}"
[1, 2, 3]                            [1, 2, 3]
arr.length                           len(arr)
arr.push(x)                          arr.append(x)
arr.slice(0, 3)                      arr[:3]
{ a: 1, b: 2 }                       {"a": 1, "b": 2}   (keys need quotes)
obj.a  /  obj["a"]                   obj["a"]           (no dot access)
if (x > 3) { ... }                   if x > 3:          (colon + indent, no braces)
for (const x of arr) { ... }         for x in arr:
function f(a, b = 2) { ... }         def f(a, b=2):
true / false / null                  True / False / None
arr.map(x => x * 2)                  [x * 2 for x in arr]
x || "fallback"                      x or "fallback"
&&  ||  !                            and  or  not
</pre>
</div>

In [ ]:
# Run through it. Change things, break things, see what happens.

name = "Ada"
scores = [7, 9, 4, 10]
student = {"name": name, "scores": scores}

print(f"{name} has {len(scores)} scores, best is {max(scores)}")
print(student["scores"][:2])

In [ ]:
# Indentation IS the syntax. Try removing the indent on the print line - Python
# will refuse to run the cell at all, which is a feature rather than a nuisance.

for score in scores:
    if score >= 9:
        print(f"{score} - excellent")
    else:
        print(f"{score} - fine")

### The one genuinely new thing: list comprehensions

JavaScript's `.map()` and `.filter()` collapse into one piece of Python syntax. You will
see this constantly for the rest of the course, so it's worth two minutes now.

In [ ]:
# JavaScript:  scores.map(s => s * 10)

[score * 10 for score in scores]

In [ ]:
# JavaScript:  scores.filter(s => s >= 9)

[score for score in scores if score >= 9]

In [ ]:
# Both at once:  scores.filter(s => s >= 9).map(s => s * 10)
#
# Read it right-to-left-ish: "for each score in scores, if it's >= 9, give me score * 10"

[score * 10 for score in scores if score >= 9]

<div style="border-left: 6px solid #900; background: #fff4f4; padding: 12px 16px; margin: 12px 0;">
<h3 style="color:#900; margin-top:0;">🎯 Now try it yourself</h3>
<p style="color:#900; margin-bottom:0;">
Given <code>words = ["balloon", "helium", "tracker", "parachute", "snow"]</code>, write
<b>three</b> list comprehensions:<br/><br/>
1. every word in capitals (<code>word.upper()</code>)<br/>
2. only the words longer than 6 letters<br/>
3. the <i>lengths</i> of only the words containing the letter "a"
</p>
</div>

In [ ]:
words = ["balloon", "helium", "tracker", "parachute", "snow"]

# 1.

# 2.

# 3.

### Functions

In [ ]:
def describe(subject, style="short", excited=False):
    """
    Build a sentence about a subject.

    The triple-quoted text you're reading is a docstring - the function's own explanation
    of itself. It's the first thing anyone reads, including you in three weeks.
    """
    ending = "!" if excited else "."
    return f"A {style} description of {subject}{ending}"

Running that printed nothing. **Defining** a function and **calling** it are different
things — you just taught Python a word, you haven't said it yet.

In [ ]:
describe("the atmosphere")

In [ ]:
# You can skip straight to a later argument by naming it. That's genuinely useful:
# it means the parameter order never has to be memorised.

describe("the atmosphere", excited=True)

## PART C: Your first model call

In [ ]:
# Our settings for today. CAPITALS is a convention meaning "set once, don't change".

CHAT_MODEL = "openai:gpt-4.1-mini"

### What is that string?

`"openai:gpt-4.1-mini"` is two things joined by a colon: **which company**, and **which
model of theirs**.

We hand it to LangChain's `init_chat_model`, which gives back an object we can talk to.
The reason we're doing it this way instead of using OpenAI's own library directly: in
Session 5 we will change that string to a different company — or to a model running on this
laptop with no internet at all — and **nothing else in our code will change.**

In [ ]:
# This builds the object. It doesn't call anything over the internet yet.

model = init_chat_model(CHAT_MODEL)
model

That cell had no `print()`. In a notebook, if the last line is just a value, the notebook
shows it to you. Get into the habit of typing a variable's name alone on a line to look
inside things — we'll do it constantly.

In [ ]:
# The first real call. Costs a fraction of a cent.

response = model.invoke("In one sentence: why does a weather balloon burst?")
response.text

<div style="border-left: 6px solid #06c; background: #f2f7ff; padding: 12px 16px; margin: 12px 0;">
<h3 style="color:#06c; margin-top:0;">🔄 No <code>await</code>?</h3>
<p style="color:#06c; margin-bottom:0;">
Correct. Python <i>can</i> do asynchronous code, but it doesn't have to, and we won't.
<code>invoke</code> simply waits for the answer and hands it back.<br/><br/>
This is one of the places Python is genuinely simpler than JavaScript, and it's why nothing
in this workshop needs promises, callbacks or <code>async</code>.
</p>
</div>

In [ ]:
# Look at the whole response object, not just the text. There's more in there than you'd think.

response

## PART D: System prompts — giving a model a job

When you talk to a chat model, each message has a **role**:

- **System** — who the model is and what its job is. Written by *you*, the programmer.
  The user never sees it.
- **Human** — what the user actually typed.

That separation is the whole reason ChatGPT can stay helpful and polite no matter what you
type at it. There's a system message you never see, telling it how to behave.

It is also, three sessions from now, exactly where we'll put the retrieved documents.

In [ ]:
# Same question, no system message

model.invoke("What is 32,000 metres?").text

In [ ]:
# Same question, with a job

messages = [
    SystemMessage(content="You are a terse flight engineer. Answer in under 15 words."),
    HumanMessage(content="What is 32,000 metres?"),
]

model.invoke(messages).text

<div style="border-left: 6px solid #900; background: #fff4f4; padding: 12px 16px; margin: 12px 0;">
<h3 style="color:#900; margin-top:0;">🎯 Now try it yourself</h3>
<p style="color:#900; margin-bottom:0;">
Write a function <code>ask(system_prompt, question)</code> that builds the two-message list
and returns the model's reply as a plain string.<br/><br/>
Then call it three times with the <b>same question</b> and three very different system
prompts — a pirate, a maths teacher, someone who only answers in questions.<br/><br/>
This function is genuinely useful and we'll keep using it. The system prompt is the single
most powerful thing you control in this entire course.
</p>
</div>

In [ ]:
def ask(system_prompt, question):
    """Ask the model a question with a given system prompt. Returns the reply as a string."""
    messages = []   # build the two-message list here
    response = None # invoke the model
    return          # return response.text

In [ ]:
# Try it with three personalities

# print(ask("...", "Why do balloons burst at high altitude?"))

<div style="border-left: 6px solid #f71; background: #fff7f0; padding: 12px 16px; margin: 12px 0;">
<h3 style="color:#f71; margin-top:0;">🚀 Extra challenge</h3>
<p style="color:#f71; margin-bottom:0;">
Try to write a system prompt that makes the model refuse to answer anything at all, no
matter what's asked.<br/><br/>
Then try to defeat your own system prompt from the human message. How hard is it? What
does that tell you about how much of a system prompt is a real rule versus a strong
suggestion?<br/><br/>
(This has a name — <b>prompt injection</b> — and it is an unsolved problem that real
security researchers are paid to work on right now.)
</p>
</div>

## Where we got to

- A working local environment, which is genuinely the hardest part of any first day
- The key stored safely, outside the code
- Python mapped onto the JavaScript you already had, including list comprehensions
- A model created from **one string** we'll later change to switch companies entirely
- System prompts: the same model, given a job, behaves completely differently

## Next session — and homework

**Bring your own documents.** Three to ten files, `.txt` or `.md`, on anything you like —
revision notes, a subject you're studying, rules of a game, a wiki you exported. They'll
become the material your assistant knows.

Next session we ask a model about material it has never seen, and watch it confidently
make things up. Then we try the obvious fix, and watch that break too.